#### This code will be used to extract x-ray image features using DenseNet loaded with pretrained cheXnet weights and the images index along with the respective features are stored in a csv file.
NOTE: All the necessary files are provided in the google drive link which can be accessed from Github repo in Readme file

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from google.colab import drive
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input

# === STEP 2: Define Paths === #
# All necessaey files are provided in the google drive link which can be accessed from readme file
CHEXNET_WEIGHTS_PATH = "CheXNet_Keras_0.3.0_weights.h5"
IMAGE_FOLDER = "chest_x_rays_folder"
SAVE_PATH = "extracted_features.csv"

# === STEP 3: Build CheXNet Model for Feature Extraction === #
base_model = DenseNet121(include_top=False, weights=None, input_shape=(224, 224, 3), pooling='avg')
x = base_model.output
output = Dense(14, activation='sigmoid', name='predictions')(x)
chexnet = Model(inputs=base_model.input, outputs=output)
chexnet.load_weights(CHEXNET_WEIGHTS_PATH)

feature_extractor = Model(inputs=chexnet.input, outputs=chexnet.get_layer('avg_pool').output)

# === STEP 4: Gather Image Paths === #
image_paths = []
for root, _, files in os.walk(IMAGE_FOLDER):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_paths.append(os.path.join(root, file))

print(f"Found {len(image_paths)} images to process.")

# === STEP 5: Extract Features === #
batch_size = 8
features_data = []

for i in tqdm(range(0, len(image_paths), batch_size), desc="Extracting Features"):
    batch = image_paths[i:i+batch_size]
    batch_images = []
    batch_filenames = []

    for path in batch:
        try:
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_AREA)
            img = np.stack([img]*3, axis=-1)
            img = preprocess_input(img.astype('float32'))
            batch_images.append(img)
            batch_filenames.append(os.path.basename(path))
        except:
            continue

    if not batch_images:
        continue

    batch_array = np.array(batch_images)
    features = feature_extractor.predict(batch_array, verbose=0)

    for j, filename in enumerate(batch_filenames):
        row = {'Image Index': filename}
        for k, val in enumerate(features[j]):
            row[f'feature_{k}'] = val
        features_data.append(row)

# === STEP 6: Save All Features === #
pd.DataFrame(features_data).to_csv(SAVE_PATH, index=False)
print(f"Feature extraction complete. Saved to: {SAVE_PATH}")
